## Lab: Building a Bigram Language Model

#### What we will build

A small language model that learns from a short conversation, then **predicts the next word**, just like a phone keyboard.

#### The idea in one line

> The model reads the training text, **counts** which word comes after which, and turns those counts into **probabilities**.

#### Our training text

A short interview between an officer and a suspect (12 lines):

```
officer: where were you last night
suspect: I was at home last night
officer: who was with you
...
```

#### What we will do

| Phase | Step | What happens |
| --- | --- | --- |
| **Training** | 1. Tokenize | Split each line into words and add `<s>` and `</s>` |
| | 2. Count words | How many times does each word appear? |
| | 3. Count pairs | How many times does each pair of words appear? |
| | 4. Probabilities | P(next \| previous) = C(previous next) / C(previous) |
| **Using the model** | 5. Predict | Type a word, and the model suggests the next one |
| **Testing** | 6–10 | Score new sentences, find the zero problem, and fix it |

In [103]:
text="""
officer: where were you last night
suspect: I was at home last night
officer: who was with you
suspect: I was alone at home
officer: you were not at home
suspect: I was at home all night
officer: did you see the car
suspect: I did not see the car
officer: where is the car now
suspect: I do not know
officer: you know where the car is
suspect: I do not know where it is
"""

In [104]:
print(text)


officer: where were you last night
suspect: I was at home last night
officer: who was with you
suspect: I was alone at home
officer: you were not at home
suspect: I was at home all night
officer: did you see the car
suspect: I did not see the car
officer: where is the car now
suspect: I do not know
officer: you know where the car is
suspect: I do not know where it is



In [105]:
import re
from collections import Counter

In [106]:
sentences = text.strip().split("\n")
sentence_tokens = [["<s>"] + re.findall(r'\w+', sentence.lower()) + ["</s>"] for sentence in sentences]
for sentence, token_list in zip(sentences, sentence_tokens):
    print(f"Sentence: {sentence}")
    print(f"Tokens: {token_list}")
    print()

Sentence: officer: where were you last night
Tokens: ['<s>', 'officer', 'where', 'were', 'you', 'last', 'night', '</s>']

Sentence: suspect: I was at home last night
Tokens: ['<s>', 'suspect', 'i', 'was', 'at', 'home', 'last', 'night', '</s>']

Sentence: officer: who was with you
Tokens: ['<s>', 'officer', 'who', 'was', 'with', 'you', '</s>']

Sentence: suspect: I was alone at home
Tokens: ['<s>', 'suspect', 'i', 'was', 'alone', 'at', 'home', '</s>']

Sentence: officer: you were not at home
Tokens: ['<s>', 'officer', 'you', 'were', 'not', 'at', 'home', '</s>']

Sentence: suspect: I was at home all night
Tokens: ['<s>', 'suspect', 'i', 'was', 'at', 'home', 'all', 'night', '</s>']

Sentence: officer: did you see the car
Tokens: ['<s>', 'officer', 'did', 'you', 'see', 'the', 'car', '</s>']

Sentence: suspect: I did not see the car
Tokens: ['<s>', 'suspect', 'i', 'did', 'not', 'see', 'the', 'car', '</s>']

Sentence: officer: where is the car now
Tokens: ['<s>', 'officer', 'where', 'is', 't

In [116]:
def get_frequency(tokens):
    frequency = dict()
    for token_list in tokens:
        for w, f in Counter(token_list).items():
            frequency[w] = frequency.get(w, 0) + f
    return frequency


frequency = get_frequency(sentence_tokens)

for token, freq in frequency.items():
    print(f"Token: {token:8}, Frequency: {freq}")
print(f"Total unique tokens - vocabulary size: {len(frequency)}")

Token: <s>     , Frequency: 12
Token: officer , Frequency: 6
Token: where   , Frequency: 4
Token: were    , Frequency: 2
Token: you     , Frequency: 5
Token: last    , Frequency: 2
Token: night   , Frequency: 3
Token: </s>    , Frequency: 12
Token: suspect , Frequency: 6
Token: i       , Frequency: 6
Token: was     , Frequency: 4
Token: at      , Frequency: 4
Token: home    , Frequency: 4
Token: who     , Frequency: 1
Token: with    , Frequency: 1
Token: alone   , Frequency: 1
Token: not     , Frequency: 4
Token: all     , Frequency: 1
Token: did     , Frequency: 2
Token: see     , Frequency: 2
Token: the     , Frequency: 4
Token: car     , Frequency: 4
Token: is      , Frequency: 3
Token: now     , Frequency: 1
Token: do      , Frequency: 2
Token: know    , Frequency: 3
Token: it      , Frequency: 1
Total unique tokens - vocabulary size: 27


In [117]:
print(text)


officer: where were you last night
suspect: I was at home last night
officer: who was with you
suspect: I was alone at home
officer: you were not at home
suspect: I was at home all night
officer: did you see the car
suspect: I did not see the car
officer: where is the car now
suspect: I do not know
officer: you know where the car is
suspect: I do not know where it is



In [118]:
def get_bigrams(tokens):
    bigrams = Counter()

    for token_list in sentence_tokens:
        bigrams.update(
            (token_list[i], token_list[i + 1])
            for i in range(len(token_list) - 1)
        )
    return bigrams

bigrams = get_bigrams(sentence_tokens)
print(f"Total unique bigrams: {len(bigrams)}")
for bigram, counts in bigrams.items():
    print(f"Bigram: {str(bigram):25}, Count: {counts}")


Total unique bigrams: 51
Bigram: ('<s>', 'officer')       , Count: 6
Bigram: ('officer', 'where')     , Count: 2
Bigram: ('where', 'were')        , Count: 1
Bigram: ('were', 'you')          , Count: 1
Bigram: ('you', 'last')          , Count: 1
Bigram: ('last', 'night')        , Count: 2
Bigram: ('night', '</s>')        , Count: 3
Bigram: ('<s>', 'suspect')       , Count: 6
Bigram: ('suspect', 'i')         , Count: 6
Bigram: ('i', 'was')             , Count: 3
Bigram: ('was', 'at')            , Count: 2
Bigram: ('at', 'home')           , Count: 4
Bigram: ('home', 'last')         , Count: 1
Bigram: ('officer', 'who')       , Count: 1
Bigram: ('who', 'was')           , Count: 1
Bigram: ('was', 'with')          , Count: 1
Bigram: ('with', 'you')          , Count: 1
Bigram: ('you', '</s>')          , Count: 1
Bigram: ('was', 'alone')         , Count: 1
Bigram: ('alone', 'at')          , Count: 1
Bigram: ('home', '</s>')         , Count: 2
Bigram: ('officer', 'you')       , Count: 2
Bigram:

In [ ]:
def get_bigrams2(tokens):
    bigrams = []
    for token_list in tokens:
        bigrams.extend([(token_list[i], token_list[i + 1]) for i in range(len(token_list) - 1)])
    return bigrams

bigrams2 = get_bigrams2(sentence_tokens)

print(f"Total bigrams: {len(bigrams)}")
print(f"Total unique bigrams: {len(set(bigrams))}")
print("------------------------------")

uniquebigrams=set(bigrams2)
i=0
for bigram in uniquebigrams:
    count = bigrams2.count(bigram)
    print(f"Bigram: {bigram}, Count: {count}")
    i+=1
    print(f"Total bigrams processed: {i}")



In [120]:
bigram_probabilities = {}
bigram_probabilities = {bigram: count / frequency[bigram[0]] for bigram, count in bigrams.items()}
for bigram, prob in bigram_probabilities.items():
    print(f"Bigram: {str(bigram):25}, Probability: {prob:.4f}")

Bigram: ('<s>', 'officer')       , Probability: 0.5000
Bigram: ('officer', 'where')     , Probability: 0.3333
Bigram: ('where', 'were')        , Probability: 0.2500
Bigram: ('were', 'you')          , Probability: 0.5000
Bigram: ('you', 'last')          , Probability: 0.2000
Bigram: ('last', 'night')        , Probability: 1.0000
Bigram: ('night', '</s>')        , Probability: 1.0000
Bigram: ('<s>', 'suspect')       , Probability: 0.5000
Bigram: ('suspect', 'i')         , Probability: 1.0000
Bigram: ('i', 'was')             , Probability: 0.5000
Bigram: ('was', 'at')            , Probability: 0.5000
Bigram: ('at', 'home')           , Probability: 1.0000
Bigram: ('home', 'last')         , Probability: 0.2500
Bigram: ('officer', 'who')       , Probability: 0.1667
Bigram: ('who', 'was')           , Probability: 1.0000
Bigram: ('was', 'with')          , Probability: 0.2500
Bigram: ('with', 'you')          , Probability: 1.0000
Bigram: ('you', '</s>')          , Probability: 0.2000
Bigram: ('

### testing our model

In [121]:
def predict_next(word):
    # Case 1: the word never appeared in training
    if word not in frequency:
        print(f"'{word}': Unknown word: no prediction.")
        return

    # Collect every bigram whose FIRST word is the input word
    candidates = {w2: p for (w1, w2), p in bigram_probabilities.items() if w1 == word}

    # Case 2: nothing ever comes after this word (e.g. </s>)
    if not candidates:
        print(f"'{word}': No next word.")
        return

    # Sort from most likely to least likely
    ordered = sorted(candidates.items(), key=lambda item: item[1], reverse=True)

    print(f"After '{word}':")
    for next_word, prob in ordered:
        print(f"    {next_word:6} {prob:.2f}")
    print(f"    sum = {sum(candidates.values()):.2f}")


w=input("Enter a word to predict the next word: ")
predict_next(w)

After 'i':
    was    0.50
    do     0.33
    did    0.17
    sum = 1.00


In [122]:
def suggestions(word):
    candidates = {w2: p for (w1, w2), p in bigram_probabilities.items() if w1 == word}
    return sorted(candidates.items(), key=lambda item: item[1], reverse=True)


def type_with_suggestions():
    END_MARKS = {".", "?", "!"}
    sentence = []

    # 1. First word, typed by the user (must be in the vocabulary)
    word = input("Start with a word: ").strip().lower()
    while word not in frequency or word in ("<s>", "</s>"):
        word = input("Unknown word. Try another: ").strip().lower()
    sentence.append(word)

    # 2. Keep suggesting the next word until the user ends the sentence
    while True:
        print("\nSentence so far:", " ".join(sentence))
        options = suggestions(word)

        if not options:
            print("The model has no next word after this.")
            break

        for i, (next_word, prob) in enumerate(options, start=1):
            print(f"  {i}. {next_word:6} {prob:.2f}")
        choice = input("Choose a number (or type . ? ! to finish): ").strip()

        if choice in END_MARKS:            # the user ends the sentence
            sentence.append(choice)
            break
        if not choice.isdigit() or not 1 <= int(choice) <= len(options):
            print("Please type a number from the list, or . ? !")
            continue

        word = options[int(choice) - 1][0]
        if word == "</s>":                 # the user picked the model's end symbol
            print("The model chose the end of the sentence.")
            break
        sentence.append(word)

    print("\nFinal sentence:", " ".join(sentence))


type_with_suggestions()


Sentence so far: i
  1. was    0.50
  2. do     0.33
  3. did    0.17

Sentence so far: i was
  1. at     0.50
  2. with   0.25
  3. alone  0.25

Sentence so far: i was at
  1. home   1.00

Sentence so far: i was at home
  1. </s>   0.50
  2. last   0.25
  3. all    0.25
The model chose the end of the sentence.

Final sentence: i was at home
